# Extended Lab: Time Series Diagnostics — Mean, Variance, Autocorrelation, White Noise & Stationarity

**Based on:** Chapter 10 concepts (time series fundamentals, ACF/PACF, cross-correlation, white noise, stationarity) extended with additional datasets, tests, and exercises beyond the original walkthrough.

**Goal of this lab:** build hands-on fluency with the diagnostic toolkit you need *before* fitting any AR/MA/ARIMA model:

1. Understand mean, variance, and ergodicity in a time-series context
2. Detect and interpret autocorrelation (ACF) and partial autocorrelation (PACF)
3. Test for white noise (Ljung-Box)
4. Detect lead/lag relationships between two series (cross-correlation)
5. Formally assess stationarity (visual, statistical, and hypothesis-test based)
6. Practice the "difference until stationary" workflow, including a case where differencing isn't the full answer (seasonality)

This notebook goes further than the original walkthrough by adding: the Augmented Dickey-Fuller (ADF) and KPSS stationarity tests, rolling statistics diagnostics, log-transform + seasonal differencing, a synthetic AR(1) vs. random-walk comparison, a synthetic leading-indicator dataset, and a capstone mini-project.

> **How to use this notebook:** Run cells top to bottom. Boxes marked **🧪 Try it yourself** are extended exercises — attempt them before checking the reference solutions notebook (`06_Solutions_TimeSeries_Diagnostics.ipynb`).


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.signal import correlate

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 4)


## Part A — Mean, Variance, and Ergodicity

A time series $X = x_1, x_2, \dots, x_t$ is a sequence of *related* samples, unlike the i.i.d. rows we work with in ordinary regression/classification. The sample mean is:

$$\bar{X} = \frac{1}{n}\sum_{t=1}^{n} x_t$$

Two process types matter for interpreting this mean:

- **Ergodic process** — a single, long realization is representative of the whole population; the sample mean converges to the true population mean as $n$ grows.
- **Non-ergodic process** — statistics computed from one stretch of the series need not match another stretch of the *same* process (e.g., a machine that needs recalibration as wear changes its output distribution).

In time-series language, the mean is often called the **signal**, and for a series to be usable in most forecasting models, that signal must be (or be transformed to be) constant over time.


In [ ]:
# A1. Simulate an ergodic process (stationary AR(1)) vs a non-ergodic-flavored process
# (a random walk with an occasional regime shift in variance)

n = 600
rng = np.random.default_rng(7)

# Stationary AR(1): x_t = 0.6*x_{t-1} + eps_t  -> ergodic-like: mean/var settle down
phi = 0.6
eps = rng.normal(0, 1, n)
ar1 = np.zeros(n)
for t in range(1, n):
    ar1[t] = phi * ar1[t-1] + eps[t]

# "Regime-shift" process: variance changes halfway through (non-ergodic flavor)
regime = np.concatenate([rng.normal(0, 1, n//2), rng.normal(0, 4, n//2)])

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(ar1); ax[0].set_title("Stationary AR(1) (ergodic-like)")
ax[1].plot(regime); ax[1].set_title("Variance regime shift (non-ergodic flavor)")
plt.tight_layout(); plt.show()

print("AR(1) mean/std, first half vs second half:",
      (ar1[:n//2].mean(), ar1[:n//2].std()), (ar1[n//2:].mean(), ar1[n//2:].std()))
print("Regime-shift mean/std, first half vs second half:",
      (regime[:n//2].mean(), regime[:n//2].std()), (regime[n//2:].mean(), regime[n//2:].std()))


**Discussion:** Notice the AR(1) series' mean/std are close between halves — a single realization is fairly representative (ergodic-like behavior). The regime-shift series has a similar mean but a very different standard deviation between halves — a statistic computed on the first half would mislead you about the second half. This is the practical meaning of ergodicity for a working analyst: *can I trust "the numbers so far" to describe "the numbers to come"?*

### 🧪 Try it yourself
Simulate a process whose **mean** (not variance) shifts partway through (e.g., a step function added to noise). Compute the mean of the first third, middle third, and last third. What does this imply about using the full-sample mean to forecast the next value?


In [ ]:
# Your code here


## Part B — The White-Noise Model & the Ljung-Box Test

White noise is the "nothing to see here" baseline: independent samples, constant (finite) variance, mean of zero. If a series is white noise, there is no pattern to model — the best forecast of the next value is just the mean. If a fitted model's *residuals* are white noise, that's a good sign the model captured the available signal.

The **Ljung-Box test** formalizes this:

- $H_0$: the data are independently distributed (no serial correlation up to lag $h$)
- $H_a$: the data show serial correlation somewhere in lags $1..h$

$$Q = n(n+2)\sum_{k=1}^{h}\frac{\hat\rho_k^2}{n-k} \;\sim\; \chi^2_h \text{ (approximately, under } H_0\text{)}$$

A small p-value (e.g., < 0.05) means we reject $H_0$ — there IS autocorrelation. A large p-value is consistent with white noise.


In [ ]:
# B1. Generate white noise and run the classic visual + statistical checks
random_white_noise = np.random.normal(loc=0, scale=1, size=1000)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(random_white_noise)
ax[0].axhline(0, color='r')
ax[0].set_title('Raw Data')
plot_acf(random_white_noise, ax=ax[1])
plt.tight_layout(); plt.show()

acorr_ljungbox(random_white_noise, lags=[10, 30, 50], return_df=True)


### 🧪 Try it yourself — is an AR(1) series white noise?

In [ ]:
# Reuse the AR(1) series 'ar1' from Part A.
# 1. Plot it and its ACF.
# 2. Run acorr_ljungbox with lags=[10, 30, 50].
# 3. In a markdown cell, state whether you reject or fail to reject H0, and why that makes sense
#    given how ar1 was constructed (x_t = 0.6*x_{t-1} + eps_t).

# Your code here


### 🧪 Try it yourself — sample-size sensitivity
Generate white noise with `size=30`, `size=200`, and `size=2000`. Run the Ljung-Box test at `lags=[10]` on each. Do you ever get a "false alarm" (a small p-value even though you *know* it's white noise)? Run it a few times with different random seeds to see how often this happens. What does this tell you about relying on a single hypothesis test at a single lag?


In [ ]:
# Your code here


## Part C — Autocorrelation (ACF) and Partial Autocorrelation (PACF)

The ACF at lag $k$ measures the raw correlation between $y_t$ and $y_{t-k}$, without controlling for the lags in between. The PACF measures the *direct* relationship between $y_t$ and $y_{t-k}$ after netting out the intermediate lags — this is what tells you the likely **AR order** to try, while the ACF's cutoff/decay pattern hints at the likely **MA order**.

We reuse the classic macro dataset (`realinv`, `realdpi`) bundled with `statsmodels`.


In [ ]:
df = sm.datasets.macrodata.load().data
df['realinv'] = round(df['realinv'].astype('float32'), 2)
df['realdpi'] = round(df['realdpi'].astype('float32'), 2)
df_mod = df[['realinv', 'realdpi']]
df_mod.head()


In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16, 8))
ax[0, 2].plot(df_mod['realinv']); ax[0, 2].set_title('Original Data (realinv)')
plot_acf(df_mod['realinv'], alpha=0.05, lags=50, ax=ax[0, 0]); ax[0, 0].set_title('Original ACF')
plot_pacf(df_mod['realinv'], alpha=0.05, lags=50, ax=ax[0, 1]); ax[0, 1].set_title('Original PACF')

diffed = np.diff(df_mod['realinv'], n=1)
ax[1, 2].plot(diffed); ax[1, 2].set_title('First-Differenced Data')
plot_acf(diffed, alpha=0.05, lags=50, ax=ax[1, 0]); ax[1, 0].set_title('Differenced ACF')
plot_pacf(diffed, alpha=0.05, lags=50, ax=ax[1, 1]); ax[1, 1].set_title('Differenced PACF')
plt.tight_layout(); plt.show()


**Reading this plot:** the original ACF decays slowly (typical of a trending / non-stationary series) — that dampening pattern hides any information about shorter-range correlation in the variance. After one first-order difference (`numpy.diff(x, n=1)`), the trend is gone and we can see a handful of ACF/PACF spikes near the short lags, which is what we'd use to propose candidate AR/MA orders in Chapter 11-style ARIMA modeling.

### 🧪 Try it yourself — realdpi
Repeat the ACF/PACF-before/after-differencing analysis for `realdpi`. Do the ACF/PACF shapes suggest similar candidate AR/MA orders to `realinv`, or different ones?


In [ ]:
# Your code here


### 🧪 Try it yourself — over-differencing
Apply a **second** first-order difference to `realinv` (i.e., difference the differenced series again). Plot the ACF/PACF. Do you see any new problems appearing (e.g., a strong negative spike at lag 1)? This is a classic symptom of *over-differencing* — differencing more than the series actually needs.


In [ ]:
# Your code here


## Part D — Cross-Correlation (CCF): Detecting Leading/Lagging Indicators

The CCF extends autocorrelation to *two* series, letting you check whether one series' current value is more related to the other series' past, present, or future values — i.e., whether one leads or lags the other.


In [ ]:
def plot_ccf(data_a, data_b, lag_lookback, percentile=95):
    """Plot a manually-normalized cross-correlation function between two 1-D series,
    with confidence bands based on the large-sample z-score approximation."""
    n = len(data_a)
    ccf = correlate(data_a - np.mean(data_a), data_b - np.mean(data_b), method='direct') / (
        np.std(data_a) * np.std(data_b) * n
    )
    _min = (len(ccf) - 1) // 2 - lag_lookback
    _max = (len(ccf) - 1) // 2 + (lag_lookback - 1)

    zscore_vals = {90: 1.645, 95: 1.96, 99: 2.576}
    z = zscore_vals[percentile] / np.sqrt(n)

    plt.figure(figsize=(13, 4))
    markers, stems, baseline = plt.stem(
        np.arange(-lag_lookback, lag_lookback - 1), ccf[_min:_max], markerfmt='o'
    )
    plt.setp(baseline, color='r', linewidth=1)
    plt.axhline(y=z, color='b', ls='--')
    plt.axhline(y=-z, color='b', ls='--')
    plt.axvline(x=0, color='black', ls='-')
    plt.title('Cross-Correlation'); plt.xlabel('Lag'); plt.ylabel('Correlation')
    plt.show()
    return ccf


In [ ]:
df_diff = pd.DataFrame()
df_diff['realinv'] = np.diff(df_mod['realinv'], n=1)
df_diff['realdpi'] = np.diff(df_mod['realdpi'], n=1)

_ = plot_ccf(data_a=df_diff['realdpi'], data_b=df_diff['realinv'], lag_lookback=50, percentile=95)


The peak sits at lag 0, so neither series leads the other in this raw comparison — and the correlation strength (~0.2) is modest. Let's build a **synthetic** example where we *know* the ground-truth lead/lag relationship, so you can build intuition for what a "true" leading indicator looks like in a CCF plot.


In [ ]:
# D1. Synthetic leading indicator: 'ad_spend' leads 'sales' by 2 weeks
rng = np.random.default_rng(3)
weeks = 300
ad_spend = rng.normal(0, 1, weeks)
sales_signal = np.roll(ad_spend, 2) * 0.8   # sales respond to ad spend from 2 weeks ago
sales = sales_signal + rng.normal(0, 0.5, weeks)
sales[:2] = rng.normal(0, 0.5, 2)  # fix wrap-around from np.roll

_ = plot_ccf(data_a=ad_spend, data_b=sales, lag_lookback=15, percentile=95)


### 🧪 Try it yourself
1. Read the CCF plot above: at which lag is the peak correlation? Does it match how the data was constructed (`ad_spend` leading `sales` by 2)?
2. Change the lag in the simulation (e.g., 2 → 5) and re-run. Confirm the CCF peak moves accordingly.
3. Using `pandas.Series.shift()`, shift `ad_spend` forward by the lag you found and re-run the CCF between the shifted `ad_spend` and `sales`. The peak should now sit at lag 0 — this is the "aligned" series you'd actually feed into a regression model.


In [ ]:
# Your code here


## Part E — Stationarity: Visual, Descriptive, and Hypothesis-Test Based

A series is (weakly/covariance) stationary if all three hold for all $t$:

1. $E[X_t] = \mu$ (constant mean)
2. $\mathrm{Var}[X_t] = \sigma^2$ (constant variance)
3. $\mathrm{Cov}(X_{t_1}, X_{t_2})$ depends only on $|t_2 - t_1|$, not on $t_1$ itself (autocorrelation depends only on lag distance)

We'll work through the classic Air Passengers dataset (monthly totals, 1949–1960).


In [ ]:
data = pd.read_csv('airline-passengers.csv', header=0, index_col=0)
data.index = pd.to_datetime(data.index, format='%Y-%m')
plt.plot(data); plt.title('Monthly Airline Passengers, 1949-1960'); plt.show()


In [ ]:
season_trend = seasonal_decompose(data, model='additive')
season_trend.plot()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
sns.boxplot(x=data.index.year, y=data['Passengers'], ax=ax, color="cornflowerblue")
ax.set(xlabel='Year', ylabel='Number of Passengers', title='Distribution of Passengers by Year')
plt.show()


The rising boxes (level) and widening spread (variance) both grow across years — a visual violation of stationarity conditions 1 and 2. Let's confirm with ACF and Ljung-Box, then move to *formal* stationarity hypothesis tests (ADF and KPSS), which the original walkthrough didn't cover.

In [ ]:
plot_acf(data, lags=20, alpha=0.05); plt.title("ACF on raw (trending) data"); plt.show()
acorr_ljungbox(data, lags=[50], return_df=True)


### E1. The Augmented Dickey-Fuller (ADF) test

- $H_0$: the series has a unit root (it is **non-stationary**)
- $H_a$: the series is stationary (or trend-stationary, depending on the regression specification)

A **small** p-value (< 0.05) → reject $H_0$ → evidence *for* stationarity.

### E2. The KPSS test

- $H_0$: the series **is** stationary (around a level or a trend)
- $H_a$: the series has a unit root (non-stationary)

Notice KPSS's null hypothesis is the *opposite* of ADF's. Using both together is a common, more robust diagnostic pattern:

| ADF says | KPSS says | Interpretation |
|---|---|---|
| stationary | stationary | Strong evidence of stationarity |
| non-stationary | non-stationary | Strong evidence of non-stationarity |
| stationary | non-stationary | Series may be trend-stationary — try detrending, not differencing |
| non-stationary | stationary | Series may need one difference; interpret with care |


In [ ]:
def stationarity_report(series, name="series"):
    series = pd.Series(series).dropna()
    adf_stat, adf_p, *_ = adfuller(series, autolag='AIC')
    kpss_stat, kpss_p, *_ = kpss(series, regression='c', nlags='auto')
    print(f"--- {name} ---")
    print(f"ADF:  stat={adf_stat:.3f}, p-value={adf_p:.4f}  -> {'stationary' if adf_p < 0.05 else 'NON-stationary'} (reject H0 if p<0.05)")
    print(f"KPSS: stat={kpss_stat:.3f}, p-value={kpss_p:.4f}  -> {'NON-stationary' if kpss_p < 0.05 else 'stationary'} (reject H0 if p<0.05)")
    print()

stationarity_report(data['Passengers'], "Airline passengers (raw)")
stationarity_report(np.diff(data['Passengers'], n=1), "Airline passengers (1st diff)")


Notice that a single first-order difference removes the *trend* but this series also has strong **seasonality** (a 12-month cycle) — differencing alone may not fully stabilize the variance, since the amplitude of the seasonal swings grows with the level (multiplicative seasonality). Two classic extra tools:

- **Log transform** to stabilize a variance that grows with the level (turns multiplicative seasonality into additive)
- **Seasonal differencing**: $Y_t' = Y_t - Y_{t-12}$ for monthly data with an annual cycle


In [ ]:
log_passengers = np.log(data['Passengers'])
seasonal_diff_of_log = log_passengers.diff(12).dropna()
first_diff_of_seasonal_diff = seasonal_diff_of_log.diff(1).dropna()

fig, ax = plt.subplots(3, 1, figsize=(10, 9))
ax[0].plot(log_passengers); ax[0].set_title('log(Passengers)')
ax[1].plot(seasonal_diff_of_log); ax[1].set_title('Seasonal diff (lag 12) of log(Passengers)')
ax[2].plot(first_diff_of_seasonal_diff); ax[2].set_title('+ first diff (log, seasonal-diff, then 1st-diff)')
plt.tight_layout(); plt.show()

stationarity_report(log_passengers, "log(Passengers)")
stationarity_report(seasonal_diff_of_log, "log + seasonal diff (12)")
stationarity_report(first_diff_of_seasonal_diff, "log + seasonal diff (12) + 1st diff")


### 🧪 Try it yourself
1. Plot the ACF of `first_diff_of_seasonal_diff` (try `lags=36`). Do you still see a spike near lag 12? What would that suggest about the seasonal order needed in a SARIMA model (covered in a later chapter)?
2. Re-run `stationarity_report` on the boxplot-by-year data restricted to just 1949-1954 vs. 1955-1960 for the *raw* series. Compare the ADF/KPSS conclusions on each half separately versus the whole series — does splitting change the conclusion?


In [ ]:
# Your code here


## Part F — Capstone Mini-Project: Full Diagnostic Pipeline on a New Series

Put everything together. Below are three synthetic candidate series. For **each**, run the complete diagnostic pipeline and write a short verdict (2-3 sentences) on: (a) is it white noise, trend-stationary, difference-stationary, or seasonal-non-stationary; (b) what transform (if any) you'd apply before modeling; (c) what AR/MA order the ACF/PACF of the (transformed) series would suggest.


In [ ]:
rng = np.random.default_rng(2024)
n = 400
t = np.arange(n)

# Series 1: pure white noise
series1 = rng.normal(0, 1, n)

# Series 2: random walk (non-stationary, no seasonality)
series2 = np.cumsum(rng.normal(0, 1, n))

# Series 3: stationary AR(2) with a mild seasonal (period 12) component added
ar2 = np.zeros(n)
e = rng.normal(0, 1, n)
for i in range(2, n):
    ar2[i] = 0.5 * ar2[i-1] - 0.3 * ar2[i-2] + e[i]
seasonal_component = 3 * np.sin(2 * np.pi * t / 12)
series3 = ar2 + seasonal_component

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(series1); ax[0].set_title('Series 1')
ax[1].plot(series2); ax[1].set_title('Series 2')
ax[2].plot(series3); ax[2].set_title('Series 3')
plt.tight_layout(); plt.show()


In [ ]:
# TODO (capstone): for each of series1, series2, series3 —
#  1. Plot the series and its ACF/PACF
#  2. Run the Ljung-Box test
#  3. Run stationarity_report() (ADF + KPSS)
#  4. If non-stationary, apply an appropriate transform (differencing / seasonal differencing) and re-test
#  5. Write your verdict as a markdown cell

# Your code here (feel free to add more cells)


## Wrap-up Questions

1. Why is it important to check for white noise *before* trying to fit an AR/MA model to a series?
2. Why does the original ACF plot in Part C decay slowly for a trending series, and why does differencing fix that?
3. In Part D, what does a CCF peak at a positive lag $k$ tell you about which series leads the other, given how `plot_ccf(data_a, data_b, ...)` is defined?
4. Give one real-world example of a series where ADF and KPSS might disagree, and explain what follow-up analysis you'd do.
5. Why might differencing alone be insufficient for the Air Passengers dataset, and what's the fix?

Continue to `06_Solutions_TimeSeries_Diagnostics.ipynb` to check your work, or `04_CheatSheet_TimeSeries_Diagnostics.ipynb` for quick syntax reference.
